# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 4: Neural Networks and LLMs

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

In [1]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR

In [2]:
load_dotenv(override=True)

True

In [3]:
LITE_MODE = True

In [4]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [13]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [14]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the HashingVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True) #Convert each product description into 5,000 numerical features, ignore common English words, and represent each feature as present/not-present.
X = vectorizer.fit_transform(documents)

In [16]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 705522 stored elements and shape (20000, 5000)>

In [17]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module): #nn.Module is PyTorch's base class for neural networks.By inheriting from it, PyTorch knows that this class contains trainable layers.
    def __init__(self, input_size): #input_size tells the network how many input features it will receive.
        super(NeuralNetwork, self).__init__() # super(...) initializes the PyTorch nn.Module part of your class.
        self.layer1 = nn.Linear(input_size, 128) #Input → 128 neurons
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x): #x is your input data.
        output1 = self.relu(self.layer1(x)) #Data goes through: x → layer1 → ReLU → output1
        output2 = self.relu(self.layer2(output1)) #output1 → layer2 → ReLU → output2
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [18]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1) #this does - y -> Convert to FloatTensor -> Add a dimension -> Shape: (number_of_samples, 1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

#create the loader
train_dataset = TensorDataset(X_train, y_train) #TensorDataset combines your input features and target values together.
train_loader = DataLoader(train_dataset, batch_size = 64, shuffle = True) #The DataLoader controls how the training data is given to the model.

#initialize the model
input_size = X_train_tensor.shape[1] #Get the number of input features
''' 
Suppose: X_train_tensor.shape is:(8000, 5000). That means:
8000 → number of samples, 5000 → number of features
shape[1] gives the second dimension: input_size = 5000
This is important because your first neural-network layer is: self.layer1 = nn.Linear(input_size, 128)
So the network knows: "I will receive 5000 features for each input."
'''
model = NeuralNetwork(input_size)

In [19]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [20]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

# We will do 2 complete runs through the data
EPOCHS = 2
for epoch in range(EPOCHS):
    model.train() #this tells pytorch The model is currently in training mode
    for batch_X, batch_y in tqdm(train_loader): #So this loop gets 64 training examples at a time.
        optimizer.zero_grad() #Before calculating new gradients, PyTorch clears the gradients from the previous batch. Why? PyTorch normally accumulates gradients.

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X) #outputs contains the model's predictions.
        loss = loss_function(outputs, batch_y)
        loss.backward() #PyTorch calculates: "How much did each weight contribute to the error?" It calculates the gradient of the loss with respect to each trainable parameter.
        optimizer.step() #Now the optimizer uses those gradients to update the model's weights.

    model.eval() #Training is finished for this epoch. Now evaluate the model
    with torch.no_grad(): #During validation, we're not updating the model.We only want to see how well it performs. So we don't need gradients. torch.no_grad() tells PyTorch not to calculate/store them, which saves memory and computation.
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)
        
    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 22189.984, Val Loss: 19939.238


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 14612.001, Val Loss: 18121.760


In [21]:
def neural_network(item): #This function uses your trained neural network to predict the price of an item from its text summary
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item() #.item() extracts the actual python number
    return max(0, result)

In [22]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$80 $69 $17 $34 $47 $171 $29 $60 $31 $62 $458 $116 $99 $173 $23 $23 $3 $13 $86 $22 $30 $33 $72 $49 $267 $237 $225 $34 $37 $40 $47 $150 $34 $24 $140 $271 $45 $138 $121 $63 $163 $91 $15 $95 $107 $61 $83 $58 $24 $32 $13 $29 $110 $24 $114 $91 $34 $115 $37 $34 $77 $10 $43 $2 $402 $95 $0 $251 $2 $241 $5 $29 $140 $106 $13 $38 $115 $47 $26 $62 $58 $126 $31 $41 $14 $53 $42 $139 $142 $125 $23 $57 $26 $11 $29 $94 $43 $45 $117 $271 $11 $65 $0 $40 $8 $68 $109 $251 $6 $52 $19 $96 $125 $31 $10 $122 $164 $51 $52 $21 $21 $211 $31 $2 $68 $16 $21 $198 $74 $48 $49 $124 $117 $27 $76 $22 $87 $71 $22 $75 $49 $143 $14 $158 $179 $66 $47 $320 $62 $12 $18 $217 $11 $58 $22 $132 $183 $17 $10 $12 $95 $11 $0 $24 $451 $17 $51 $12 $33 $36 $12 $23 $234 $55 $16 $6 $20 $8 $36 $132 $359 $17 $80 $41 $47 $84 $45 $34 $16 $10 $68 $69 $66 $4 $12 $25 $93 $36 $1 $14 

# And now - to the frontier!

Let's see how Frontier models do out of the box; no training, just inference based on their world knowledge.

Tomorrow we will do some training.

In [23]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [24]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [25]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [26]:
def gpt_oss(item):
    response = completion(model = "groq/openai/gpt-oss-20b", messages = messages_for(item))
    return response.choices[0].message.content

In [27]:
gpt_oss(test[0])

'$520'

In [28]:
test[0].price

219.0

In [29]:
evaluate(gpt_oss, test, size = 30, workers = 1)

  0%|          | 0/30 [00:00<?, ?it/s]

$31 $94 $23 $0 $20 $30 $176 $135 $1 $620 $463 $20 $230 $19 $41 $4 $101 $30 $115 $1 $79 $56 $75 $275 $252 $253 $145 $2 $81 $55 

In [30]:
def gpt_oss_2(item):
    response = completion(model = "groq/openai/gpt-oss-120b", messages = messages_for(item))
    return response.choices[0].message.content

In [31]:
gpt_oss_2(test[0])

'$249'

In [32]:
evaluate(gpt_oss_2, test, size = 30, workers = 1)

  0%|          | 0/30 [00:00<?, ?it/s]

$0 $17 $31 $49 $119 $111 $64 $36 $9 $869 $528 $20 $40 $6 $40 $8 $10 $28 $11 $20 $69 $124 $85 $224 $212 $274 $404 $0 $100 $65 

In [35]:
def gemini(item):
    response = completion(model = "gemini/gemini-flash-lite-latest", messages = messages_for(item))
    return response.choices[0].message.content

In [36]:
gemini(test[0])


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



'$229'


Provider List: https://docs.litellm.ai/docs/providers



In [ ]:
evaluate(gemini, test, size = 30, workers = 1)